In [18]:
import numpy as np
import pandas as pd
import plotly
import scipy
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [19]:
df = pd.read_json("Master.json")
#curvature_df = pd.read_json("CurcuitCurvatures.json")
#df = pd.merge(master_df, curvature_df, on="circuit_key", how="left")
df.columns

Index(['session_key', 'driver_number', 'date_start', 'lap_duration',
       'air_temperature', 'track_temperature', 'humidity', 'rainfall',
       'meeting_key', 'circuit_key', 'circuit_short_name', 'session_name',
       'session_type', 'team_name', 'position', 'has_grid_position'],
      dtype='str')

In [20]:
df

,session_key,driver_number,date_start,lap_duration,air_temperature,track_temperature,humidity,rainfall,meeting_key,circuit_key,circuit_short_name,session_name,session_type,team_name,position,has_grid_position
0,7765,2,2023-03-03T11:30:07.574Z,NaN,27.400000,41.900000,11.0,0.0,1141,63,Sakhir,Practice 1,Practice,Williams,0,False
1,7765,77,2023-03-03T11:30:09.215Z,129.966,27.400000,41.850000,11.0,0.0,1141,63,Sakhir,Practice 1,Practice,Alfa Romeo,0,False
2,7765,24,2023-03-03T11:30:14.011Z,140.357,27.400000,41.850000,11.0,0.0,1141,63,Sakhir,Practice 1,Practice,Alfa Romeo,0,False
3,7765,4,2023-03-03T11:30:16.168Z,128.935,27.400000,41.850000,11.0,0.0,1141,63,Sakhir,Practice 1,Practice,McLaren,0,False
4,7765,81,2023-03-03T11:30:26.277Z,142.794,27.366667,41.833333,11.0,0.0,1141,63,Sakhir,Practice 1,Practice,McLaren,0,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
228207,11342,10,2026-07-26T14:42:42.780Z,85.702,31.300000,46.300000,26.6,0.0,1291,4,Hungaroring,Race,Race,Alpine,12,True
228208,11342,6,2026-07-26T14:42:46.272Z,84.122,31.300000,46.300000,26.6,0.0,1291,4,Hungaroring,Race,Race,Red Bull Racing,8,True
228209,11342,63,2026-07-26T14:42:48.774Z,83.759,31.300000,46.300000,26.6,0.0,1291,4,Hungaroring,Race,Race,Mercedes,6,True
228210,11342,18,2026-07-26T14:42:53.100Z,85.750,31.300000,46.300000,26.6,0.0,1291,4,Hungaroring,Race,Race,Aston Martin,20,True


In [21]:
df.dtypes

session_key             int64
driver_number           int64
date_start                str
lap_duration          float64
air_temperature       float64
track_temperature     float64
humidity              float64
rainfall              float64
meeting_key             int64
circuit_key             int64
circuit_short_name        str
session_name              str
session_type              str
team_name                 str
position                int64
has_grid_position        bool
dtype: object

In [22]:
df = df.dropna()

In [23]:
df

,session_key,driver_number,date_start,lap_duration,air_temperature,track_temperature,humidity,rainfall,meeting_key,circuit_key,circuit_short_name,session_name,session_type,team_name,position,has_grid_position
1,7765,77,2023-03-03T11:30:09.215Z,129.966,27.400000,41.850000,11.0,0.0,1141,63,Sakhir,Practice 1,Practice,Alfa Romeo,0,False
2,7765,24,2023-03-03T11:30:14.011Z,140.357,27.400000,41.850000,11.0,0.0,1141,63,Sakhir,Practice 1,Practice,Alfa Romeo,0,False
3,7765,4,2023-03-03T11:30:16.168Z,128.935,27.400000,41.850000,11.0,0.0,1141,63,Sakhir,Practice 1,Practice,McLaren,0,False
4,7765,81,2023-03-03T11:30:26.277Z,142.794,27.366667,41.833333,11.0,0.0,1141,63,Sakhir,Practice 1,Practice,McLaren,0,False
5,7765,21,2023-03-03T11:30:39.042Z,124.529,27.350000,41.800000,11.0,0.0,1141,63,Sakhir,Practice 1,Practice,AlphaTauri,0,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
228207,11342,10,2026-07-26T14:42:42.780Z,85.702,31.300000,46.300000,26.6,0.0,1291,4,Hungaroring,Race,Race,Alpine,12,True
228208,11342,6,2026-07-26T14:42:46.272Z,84.122,31.300000,46.300000,26.6,0.0,1291,4,Hungaroring,Race,Race,Red Bull Racing,8,True
228209,11342,63,2026-07-26T14:42:48.774Z,83.759,31.300000,46.300000,26.6,0.0,1291,4,Hungaroring,Race,Race,Mercedes,6,True
228210,11342,18,2026-07-26T14:42:53.100Z,85.750,31.300000,46.300000,26.6,0.0,1291,4,Hungaroring,Race,Race,Aston Martin,20,True


In [24]:
print(df["session_type"].value_counts(), "\n")
print(df["position"].value_counts())

session_type
Race          106266
Practice       85459
Qualifying     27994
Name: count, dtype: int64 

position
0     113453
6       5630
4       5576
14      5544
10      5482
7       5475
13      5463
12      5345
16      5328
17      5327
3       5322
2       5317
15      5298
5       5267
11      5221
1       5205
8       5155
9       5084
19      4875
18      4809
20      4451
21       616
22       476
Name: count, dtype: int64


> We see there is a class imbalance problem when it comes to positions. We must keep in mind that ALL practice sessions are labeled as position 0. They provide important lap-time information, but position information is moot for this data.

In [25]:
# Drop unique identifiers
df_cleansed = df.drop(columns=["session_key", "meeting_key", "circuit_key", "session_name"])

# Drop time, as it is not relevant for unseen races and introduces noise
df_cleansed = df_cleansed.drop(columns=["date_start"])

# onehot encode categorical variables
categorical_columns = ["team_name", "driver_number", "session_type", "circuit_short_name"]
df_cleansed = pd.get_dummies(df_cleansed, columns=categorical_columns)

X = df_cleansed.drop(columns=["lap_duration"])
y = df_cleansed[["lap_duration", "session_type_Practice", "session_type_Race"]]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.11, random_state=42) # ~10% of the data is used for validation

# Create X and Y validation and test sets for non-practice sessions
X_val_non_practice = X_val[X_val["session_type_Practice"] == 0]
y_val_non_practice = y_val[y_val["session_type_Practice"] == 0]

X_test_non_practice = X_test[X_test["session_type_Practice"] == 0]
y_test_non_practice = y_test[y_test["session_type_Practice"] == 0]

# Create X and Y validation and test sets for racing sessions

X_val_racing = X_val[X_val["session_type_Race"] == 1]
y_val_racing = y_val[y_val["session_type_Race"] == 1]

X_test_racing = X_test[X_test["session_type_Race"] == 1]
y_test_racing = y_test[y_test["session_type_Race"] == 1]

# remove the session_type_Practice column from Y features, as it is not the target variable
y_train = y_train.drop(columns=["session_type_Practice", "session_type_Race"])
y_val = y_val.drop(columns=["session_type_Practice", "session_type_Race"])
y_test = y_test.drop(columns=["session_type_Practice", "session_type_Race"])
y_val_non_practice = y_val_non_practice.drop(columns=["session_type_Practice", "session_type_Race"])
y_test_non_practice = y_test_non_practice.drop(columns=["session_type_Practice", "session_type_Race"])
y_val_racing = y_val_racing.drop(columns=["session_type_Practice", "session_type_Race"])
y_test_racing = y_test_racing.drop(columns=["session_type_Practice", "session_type_Race"])

In [26]:
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)

In [27]:
rf_model.fit(X_train, y_train)

c:\Users\gabri\Documents\Skool\F1-Grand-Prix-Predictor\venv\Lib\site-packages\sklearn\base.py:1403: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease o

In [38]:
from sklearn.metrics import mean_absolute_error

# Validation scores | can't do accuracy due to multi-output
rf_preds = rf_model.predict(X_val)
rf_score = mean_absolute_error(y_val, rf_preds)


# Non-practice validation scores
rf_preds_non_practice = rf_model.predict(X_val_non_practice)
rf_score_non_practice = mean_absolute_error(y_val_non_practice, rf_preds_non_practice)

# Racing validation scores
rf_preds_racing = rf_model.predict(X_val_racing)
rf_score_racing = mean_absolute_error(y_val_racing, rf_preds_racing)

print(f"Random Forest Validation MAE: {rf_score:.4f}")
print(f"Random Forest Validation MAE (non-practice sessions): {rf_score_non_practice:.4f}")
print(f"Random Forest Validation MAE (racing sessions): {rf_score_racing:.4f}")


Random Forest Validation MAE: 27.0958
Random Forest Validation MAE (non-practice sessions): 12.0287
Random Forest Validation MAE (racing sessions): 1.8362


In [ ]:
# test scores



rf_test_preds = rf_model.predict(X_test)
rf_test_mae = mean_absolute_error(y_test, rf_test_preds)

#non=practice test scores
rf_test_preds_non_practice = rf_model.predict(X_test_non_practice)
rf_test_mae_non_practice = mean_absolute_error(y_test_non_practice, rf_test_preds_non_practice)

# racing test scores
rf_test_preds_racing = rf_model.predict(X_test_racing)
rf_test_mae_racing = mean_absolute_error(y_test_racing, rf_test_preds_racing)

print(f"Random Forest Test predictions: {rf_test_preds}")
print(f"Random Forest Test MAE: {rf_test_mae:.4f}")

Random Forest Test predictions: [ 86.03015  95.12305 129.23897 ...  87.95059 106.9986  101.70869]
Random Forest Test MAE: 26.6961


In [ ]:
#All zero and one predictions for comparison

all_zero = [0] * len(rf_test_preds)
all_one = [1] * len(rf_test_preds)

all_zero_acc = mean_absolute_error(y_test, all_zero)
all_one_acc = mean_absolute_error(y_test, all_one)

# all zero and one predictions for non-practice sessions

all_zero_non_practice = [0] * len(rf_test_preds_non_practice)
all_one_non_practice = [1] * len(rf_test_preds_non_practice)

all_one_acc_non_practice = mean_absolute_error(y_test_non_practice, all_one_non_practice)
all_zero_acc_non_practice = mean_absolute_error(y_test_non_practice, all_zero_non_practice)


# all zero and one predictions for racing sessions

all_zero_racing = [0] * len(rf_test_preds_racing)
all_one_racing = [1] * len(rf_test_preds_racing)

all_one_acc_racing = mean_absolute_error(y_test_racing, all_one_racing)
all_zero_acc_racing = mean_absolute_error(y_test_racing, all_zero_racing)

print("====Test Accuracy Comparison====")
print(f"All zero Accuracy: {all_zero_acc:.4f}")
print(f"All zero Non-Practice Accuracy: {all_zero_acc_non_practice:.4f}")
print(f"All zero Racing Accuracy: {all_zero_acc_racing:.4f}")
print()
print(f"All one Accuracy: {all_one_acc:.4f}")
print(f"All one Non-Practice Accuracy: {all_one_acc_non_practice:.4f}")
print(f"All one Racing Accuracy: {all_one_acc_racing:.4f}")
print()
print(f"Random Forest Accuracy: {rf_test_mae:.4f}")
print(f"Random Forest Non-Practice Accuracy: {rf_test_mae_non_practice:.4f}")
print(f"Random Forest Racing Accuracy: {rf_test_mae_racing:.4f}")


====Test Accuracy Comparison====
All zero Accuracy: 130.8658
All zero Non-Practice Accuracy: 112.9429
All zero Racing Accuracy: 95.2692

All one Accuracy: 129.8658
All one Non-Practice Accuracy: 111.9429
All one Racing Accuracy: 94.2692

Random Forest Accuracy: 26.6961
Random Forest Non-Practice Accuracy: 11.2933
Random Forest Racing Accuracy: 1.8152


Due to the relatively high accuracy of guessing all ones for racing, our data may have a class imbalance problem